In [5]:
from pathlib import Path
import xarray as xr
import sys
import numpy as np
sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [6]:
Processed_Dataset = "20230930_PPI"

In [7]:
# ---- scan configuration -------------------------------------------------
SCAN_TYPE = "ppi"          # "zenith" | "ppi" | "rhi"

if SCAN_TYPE == "zenith":
    Dataset_94GHz     = "20230930_94GHz"
    Dataset_35GHz     = "20230930_35GHz"
    Processed_Dataset = "20230930"
    SCAN_TOKEN        = "ZEN"
    ELEV_DEG          = 90.0
    TIME_AVERAGE_S    = 30.0          # rolling mean over profiles is legal
else:
    Dataset_94GHz     = "20230930_94GHz_PPI"
    Dataset_35GHz     = "20230930_35GHz_PPI"
    Processed_Dataset = "20230930_PPI"
    SCAN_TOKEN        = "PPI"         # <-- replace with the answer from 0.1
    ELEV_DEG          = 75.0          # <-- replace with the answer from 0.2
    TIME_AVERAGE_S    = 0.0           # each ray stands alone

SIN_ELEV = np.sin(np.radians(ELEV_DEG))
print(f"SCAN_TYPE={SCAN_TYPE}  elev={ELEV_DEG} deg  sin(elev)={SIN_ELEV:.4f}")

SCAN_TYPE=ppi  elev=75.0 deg  sin(elev)=0.9659


In [8]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [9]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [10]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 14993, range: 339)> Size: 41MB
array([[0.43668062, 0.42980249, 0.42435358, ...,        nan,        nan,
               nan],
       [0.43671894, 0.42984031, 0.424391  , ...,        nan,        nan,
               nan],
       [0.43671894, 0.42984031, 0.424391  , ...,        nan,        nan,
               nan],
       ...,
       [0.61707634, 0.60818642, 0.60004468, ...,        nan,        nan,
               nan],
       [0.61712621, 0.60823567, 0.60009336, ...,        nan,        nan,
               nan],
       [0.61712621, 0.60823567, 0.60009336, ...,        nan,        nan,
               nan]], shape=(14993, 339))
Coordinates:
  * Time     (Time) datetime64[ns] 120kB 2023-09-30T20:00:52 ... 2023-10-01T0...
  * range    (range) float32 1kB 111.8 134.2 156.5 ... 1.194e+04 1.197e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 14993, range:

In [11]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")